# MLP S1 — Multi-Source Store Sales: starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s1-store-sales.ipynb)

A retail chain runs five stores. Four of them recorded how many units of each
item they sold; the fifth, **Neighborhood_Market**, did not. The task is to
predict `quantity_sold` for its 409 items. No single source holds everything:

| Section | Source | What it provides |
|---|---|---|
| 3 | one file per store: CSV, Excel, JSON | item features and `quantity_sold` |
| 4 | a REST API protected by a password | `unit_cost` |
| 5 | a web page rendered by JavaScript | `customer_score`, `total_reviews` |

Section 6 predicts Neighborhood_Market and submits. Section 7 goes further
with SQL: the course PostgreSQL database adds `weekly_footfall`, a
description of each store.

Challenge page: [https://ml-arena.com/viewchallenge/190](https://ml-arena.com/viewchallenge/190).
**The bar is a score of −20 or higher**: the score is −MAE, so your
predictions must be off by at most 20 units per item on average.

Cells marked **TODO** are yours to write; every other cell runs as it is.
Each TODO cell is followed by a cell that checks it, so an error shows up
where it was made.

---

## 1. Setup

The notebook needs one secret, `MLARENA_API_KEY`: ML-Arena, Profile → API
Keys (it starts with `mlk_user_`). Never paste its value into a cell: a
notebook is shared with its code and its outputs.

In Colab, add it to the *Secrets* panel (the key icon on the left) and allow
this notebook to access it. Outside Colab, set it in your environment before
starting Jupyter.

In [16]:
import os
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mlarena-sdk", "selenium", "openpyxl",
                    "beautifulsoup4"], check=True)

import mlarena
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

In [17]:
MLARENA_API_KEY = "mlk_user_58d09d93c29c9d77_df10ea8e23d3f3f406a4393e008a6028"

CHALLENGE_ID = 190
client = mlarena.connect(api_key=MLARENA_API_KEY)
print("Running in Colab:", IN_COLAB)

Running in Colab: True


---

## 2. Your first push

Submit something before modelling anything. A constant prediction proves the
whole pipeline: the files download, the submission has the right format, and
your key works. It will not clear the bar.

The first cell downloads the five files of the challenge into the working
directory.

In [18]:
for path in client.download_dataset(CHALLENGE_ID, "."):
    print(path)

./CityMart_data.csv
./SuperSaver_Outlet_data.xlsx
./HighStreet_Bazaar_data.json
./Neighborhood_Market_data.csv
./Greenfield_Grocers_data.csv


In [19]:
test = pd.read_csv("Neighborhood_Market_data.csv")
train = pd.read_csv("CityMart_data.csv")   # one store is enough for now

submission = pd.DataFrame({
    "item_code": test["item_code"],
    "quantity_sold": train["quantity_sold"].mean(),
})
submission.to_csv("submission.csv", index=False)
submission.head()

,item_code,quantity_sold
0,P0002,217.66506
1,P0004,217.66506
2,P0005,217.66506
3,P0010,217.66506
4,P0013,217.66506


`client.submit` uploads `submission.csv` and starts the scoring.
`wait_for_score` then waits for the result and returns the submission's row
of the leaderboard, or raises with the platform's message if the file was
rejected. Scoring takes about 20 seconds once the submission is queued.

In [20]:
def wait_for_score(result, timeout=600):
    """Wait until a submission is scored; return its leaderboard row."""
    agent_id = result["agent_id"]
    deadline = time.time() + timeout
    while time.time() < deadline:
        status = client.agent_status(CHALLENGE_ID, agent_id)
        if status["status"] in ("upload_failed", "deploy_failed"):
            raise RuntimeError(status["last_status_message"])
        if status["status"] == "active":
            board = client.leaderboard(CHALLENGE_ID)
            row = board[board["agentAttachId"] == agent_id]
            if len(row):
                return row[["Rank", "AgentName", "MeanReward", "MeanReward2"]
                           ].rename(columns={"MeanReward": "score (-MAE)",
                                             "MeanReward2": "RMSE"})
        time.sleep(10)
    raise TimeoutError(f"submission {agent_id} not scored in {timeout} s")


result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"],
                       agent_name="first-push")
wait_for_score(result)

,Rank,AgentName,score (-MAE),RMSE
46,23,first-push,-26.369329,32.705896


The row is on the leaderboard of the challenge page too. The rest of the
notebook replaces the constant with a model, one source at a time.

---

## 3. Files

Each store's export was written by a different system: the separator, the
header row, the column names and the sheet structure differ. Look at the raw
file before choosing the options of the pandas reader. `peek` prints the
first lines of a text file with `repr`, which makes separators and empty
fields visible.

Use `item_code` as the index of every table: the joins of the next sections
align rows on it. Once read, every training table has the columns in
`FILE_COLS` plus `quantity_sold`, and some also have `last_modified`.

In [21]:
FILE_COLS = ["store_name", "mass", "dimension_length", "dimension_width",
             "dimension_height", "days_since_last_purchase",
             "package_volume", "stock_age"]
TARGET = "quantity_sold"


def peek(name, n=5, width=160):
    """Print the first raw lines of a text file, separators visible."""
    with open(name, encoding="utf-8") as f:
        for _ in range(n):
            print(repr(f.readline()[:width]))

### CityMart — `CityMart_data.csv`

Hint: a standard CSV; the defaults of `read_csv` fit.

In [23]:
peek("CityMart_data.csv")
# TODO: read the file with item_code as the index.
df_citymart = pd.read_csv("CityMart_data.csv", index_col="item_code")


'item_code,store_name,mass,dimension_length,dimension_width,dimension_height,days_since_last_purchase,package_volume,stock_age,quantity_sold,last_modified\n'
'P0019,CityMart,2.81,26.83,38.75,24.89,323,25877.199624999997,253,202,2023-01-19\n'
'P0024,CityMart,3.3,59.23,34.99,21.78,321,45138.128706,21,225,2023-01-24\n'
'P0025,CityMart,2.34,22.6,16.9,60.12,291,22962.232799999998,316,278,2023-01-25\n'
'P0034,CityMart,6.54,18.59,68.72,21.99,126,28092.330551999996,612,233,2023-02-03\n'


### Greenfield_Grocers — `Greenfield_Grocers_data.csv`

Hint: separated by `|`, with lines of empty fields above the header,
upper-case column names and empty columns at the end of each line.

In [26]:
peek("Greenfield_Grocers_data.csv")
# TODO: read the file, drop the empty columns, and give the columns and the
# index the same lower-case names as CityMart.
df_greenfield = pd.read_csv("Greenfield_Grocers_data.csv", sep="|", skiprows=3, index_col="ITEM_CODE")
df_greenfield.columns = df_greenfield.columns.str.lower()
df_greenfield.index.name = df_greenfield.index.name.lower()
df_greenfield.dropna(axis=1, how='all', inplace=True)

'||||||||||||\n'
'||||||||||||\n'
'||||||||||||\n'
'ITEM_CODE|STORE_NAME|MASS|DIMENSION_LENGTH|DIMENSION_WIDTH|DIMENSION_HEIGHT|DAYS_SINCE_LAST_PURCHASE|PACKAGE_VOLUME|STOCK_AGE|QUANTITY_SOLD|LAST_MODIFIED|1|\n'
'P0006|Greenfield_Grocers|5.02|86.68|71.64|16.42|66|101964.18038400002|450|130|2023-01-06||\n'


### SuperSaver_Outlet — `SuperSaver_Outlet_data.xlsx`

Hint: a workbook with two sheets, `Quantity` (the target) and `Info` (the
features); `sheet_name=None` reads both. Compare `Info`'s header with its
first rows: the names are not above the values they describe.

In [29]:
# TODO: read both sheets, fix the column names of Info, and join the two
# sheets on item_code (not on row position).
excel_sheets = pd.read_excel("SuperSaver_Outlet_data.xlsx", sheet_name=None)

df_quantity = excel_sheets['Quantity'].set_index('item_code')

# Process the 'Info' sheet:
# Read without header, as no proper header row seems to exist
df_info = pd.read_excel("SuperSaver_Outlet_data.xlsx", sheet_name='Info', header=None)

# Define column names based on the expected structure and observed data.
# The file has 10 columns, the last one is expected to be empty.
new_info_columns = ['item_code'] + FILE_COLS + ['extra_empty_column']
df_info.columns = new_info_columns

# Drop the extra empty column (if any)
df_info.dropna(axis=1, how='all', inplace=True)

# Set 'item_code' as the index. Column names are already lowercase by explicit assignment.
df_info = df_info.set_index('item_code')

# Join the two dataframes on item_code
df_supersaver = df_quantity.join(df_info, how='left')

### HighStreet_Bazaar — `HighStreet_Bazaar_data.json`

Hint: a list of JSON records, one object per item. Its `last_modified` is a
Unix time in milliseconds; the model does not use that column.

In [32]:
peek("HighStreet_Bazaar_data.json", n=1)
# TODO: read the records with item_code as the index.
df_highstreet = pd.read_json("HighStreet_Bazaar_data.json").set_index('item_code')

'[{"item_code":"P0001","store_name":"HighStreet_Bazaar","mass":6.11,"dimension_length":75.46,"dimension_width":91.62,"dimension_height":92.08,"days_since_last_pu'


### Aggregate the four stores

In [34]:
# TODO: stack the four tables into one table, `train`.
train = pd.concat([df_citymart, df_greenfield, df_supersaver, df_highstreet])

In [ ]:
expected = set(FILE_COLS + [TARGET, "last_modified"])
assert set(train.columns) == expected, set(train.columns) ^ expected
assert len(train) == 1591, f"{len(train)} rows, expected 1591"
assert train.index.name == "item_code" and train.index.is_unique
assert train[TARGET].notna().all()
for col in FILE_COLS[1:] + [TARGET]:
    assert pd.api.types.is_numeric_dtype(train[col]), f"{col} is not numeric"

print(train["store_name"].value_counts())
train.isna().sum()[lambda s: s > 0]

### A baseline to measure each source by

`get_simple_baseline` comes from the original exercise. It drops the
columns that are not features, fills missing values with -1, standardises
the features, fits a linear regression and returns the mean absolute error
(MAE, in units sold) over 5-fold cross-validation. With `X_data_test` it
also refits on every row and returns predictions for `X_data_test`.

The model stays the same in every section, so a change in MAE comes from the
data that was added. `store_name` and `last_modified` are not features: a
store name never seen in training says nothing about Neighborhood_Market,
and a date needs feature engineering before a linear model can use it.

In [ ]:
DROP = ["store_name", "last_modified"]


def get_simple_baseline(data, fillna_value=-1, drop_cols=None, k_fold=5,
                        target_col=TARGET, X_data_test=None):
    data = data.drop(columns=drop_cols or []).fillna(fillna_value)
    y = data[target_col]
    X = data.drop(columns=[target_col])

    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    if X_data_test is not None:
        X_data_test = X_data_test.drop(columns=drop_cols or [])
        X_data_test = scaler.transform(X_data_test.fillna(fillna_value))

    model = LinearRegression()
    kf = KFold(n_splits=k_fold, shuffle=True, random_state=42)
    scores = []
    for train_index, test_index in kf.split(X):
        model.fit(X[train_index], y.iloc[train_index])
        y_pred = model.predict(X[test_index])
        scores.append(mean_absolute_error(y.iloc[test_index], y_pred))

    if X_data_test is not None:
        model.fit(X, y)
        return np.mean(scores), model.predict(X_data_test)
    return np.mean(scores)


cv_mae = {"files": get_simple_baseline(train, drop_cols=DROP)}
print(f"CV MAE, files: {cv_mae['files']:.2f}")

---

## 4. API — `unit_cost`

The documentation is at
<https://www.raphaelcousin.com/module4/api-doc>, under the exercise
endpoints. Access takes two calls: `{API}/auth` returns a password in its
`data`, and `{API}/<password>/prices` returns `{item_code: unit_cost}`.

This site answers an unknown path, a wrong password included, with its HTML
home page and status 200. `get_api` refuses an HTML response and checks the
`status` field of the reply, so a failed call stops at the call.

In [ ]:
API = "https://www.raphaelcousin.com/api/exercise"


def get_api(url):
    """GET an endpoint of the course API and return its `data` field."""
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    if r.headers["Content-Type"].startswith("text/html"):
        raise ValueError(f"{url} returned an HTML page, not JSON")
    body = r.json()
    if body["status"] != "success":
        raise ValueError(f"{url}: {body['message']}")
    print(body["message"])
    return body["data"]

Hint: request the password at run time and keep it in a variable; do not
print it. `pd.DataFrame.from_dict(..., orient="index")` turns a dict keyed
by `item_code` into one row per item. A left join on the index keeps every
row of `train`.

In [ ]:
# TODO: get the password, then the prices; build df_prices (index
# item_code, one column unit_cost) and join it to train.
password = ...
df_prices = ...
train = ...

In [ ]:
assert len(train) == 1591, "the join changed the number of rows"
assert train["unit_cost"].notna().all(), "some items have no unit_cost"

cv_mae["+ API"] = get_simple_baseline(train, drop_cols=DROP)
print(f"CV MAE, files + API: {cv_mae['+ API']:.2f}")

---

## 5. Web scraping — `customer_score` and `total_reviews`

The page <https://www.raphaelcousin.com/module4/scrapable-data> is built in
the browser by JavaScript: `requests.get` returns the page before the tables
exist. Selenium drives a real browser, Chrome without a window.

Colab has no Chrome. The next cell installs Google's Debian package of
Chrome; on the first `webdriver.Chrome(...)`, Selenium downloads the
matching chromedriver. Outside Colab the cell does nothing and your own
Chrome is used.

In [ ]:
if IN_COLAB:
    deb = "google-chrome-stable_current_amd64.deb"
    apt_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
    subprocess.run(["wget", "-q",
                    f"https://dl.google.com/linux/direct/{deb}"],
                   check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", f"./{deb}"],
                   check=True, env=apt_env)

`--headless=new` runs without a display, `--no-sandbox` lets Chrome start as
`root` (Colab's user), and `--disable-dev-shm-usage` avoids the small
`/dev/shm` of containers. `WebDriverWait` waits for the first table row
instead of sleeping for a fixed time.

In [ ]:
URL = "https://www.raphaelcousin.com/module4/scrapable-data"

options = webdriver.ChromeOptions()
for arg in ("--headless=new", "--no-sandbox", "--disable-dev-shm-usage"):
    options.add_argument(arg)

driver = webdriver.Chrome(options=options)
try:
    driver.get(URL)
    WebDriverWait(driver, 30).until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "table tbody tr")))
    html = driver.page_source
finally:
    driver.quit()

soup = BeautifulSoup(html, "html.parser")
tables = {tuple(th.text for th in t.find_all("th")): t
          for t in soup.find_all("table")}
for headers, table in tables.items():
    print(headers, len(table.find("tbody").find_all("tr")), "rows")

Hint: `tables` maps each table's header row to the table. Take the one whose
headers are *Item Code, Customer Score, Total Reviews, Updated Timestamp* by
its headers, not by its position on the page. Each `tr` of its `tbody` is one
item and each `td` one cell. Cell text is a string: convert the numbers.
*Updated Timestamp* changes at random on every page load, so it describes
the page, not the item: leave it out.

In [ ]:
# TODO: build df_scores (index item_code, numeric columns customer_score and
# total_reviews) from the Exercise table, and join it to train.
df_scores = ...
train = ...

In [ ]:
SCRAPED = ["customer_score", "total_reviews"]
assert len(train) == 1591, "the join changed the number of rows"
assert train[SCRAPED].notna().all().all(), "some items have no score"
assert all(pd.api.types.is_numeric_dtype(train[c]) for c in SCRAPED)

cv_mae["+ scraping"] = get_simple_baseline(train, drop_cols=DROP)
print(f"CV MAE, files + API + scraping: {cv_mae['+ scraping']:.2f}")

---

## 6. Predict Neighborhood_Market and submit

`Neighborhood_Market_data.csv` has CityMart's layout without
`quantity_sold`. Add the columns of the API and of the page to it, the same
way as for `train`.

In [ ]:
# TODO: read Neighborhood_Market_data.csv with item_code as the index, and
# add unit_cost, customer_score and total_reviews.
test = ...

The next cell puts the columns of `test` in the order of `train` (a
`KeyError` names any column that is missing), refits the baseline on the
four stores and writes `submission.csv`.

In [ ]:
test = test[train.columns.drop(TARGET)]
assert len(test) == 409 and test.index.is_unique
added = ["unit_cost", "customer_score", "total_reviews"]
assert test[added].notna().all().all(), "a joined source left a gap"

cv, pred = get_simple_baseline(train, drop_cols=DROP, X_data_test=test)
submission = pd.DataFrame({"item_code": test.index, TARGET: pred})
assert submission[TARGET].notna().all()
submission.to_csv("submission.csv", index=False)
print(f"CV MAE: {cv:.2f}")
submission.describe()

In [ ]:
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"],
                       agent_name="three-sources")
row = wait_for_score(result)
score = float(row["score (-MAE)"].iloc[0])
print("clears the bar" if score >= -20 else "below the bar (-20)")
row

---

## 7. Going further — the database

Every feature so far describes an item. The model is trained on four stores
and applied to a fifth it has never seen, so nothing tells it how busy that
store is. The course PostgreSQL database describes the stores; this section
reads it with SQL.

The connection string, `DATABASE_URL`, is in the *Access — today's sandbox*
section of the Lab 1.1 page. It connects with a read-only account. Add it
as a second secret, the same way as the API key. The schema `retail`
describes the data of this challenge; start with its data dictionary, which
says what each column means and which source it comes from.

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "psycopg[binary]"], check=True)
    os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")

from sqlalchemy import create_engine

engine = create_engine(os.environ["DATABASE_URL"])
pd.read_sql("SELECT * FROM retail.data_dictionary ORDER BY source, "
            "column_name", engine)

In [ ]:
stores = pd.read_sql("SELECT * FROM retail.stores ORDER BY store_name",
                     engine)
stores

Hint: `weekly_footfall` is a store-level feature, and it matters precisely
because the test store differs from the training stores. Match it to each
item on `store_name`, for example with `Series.map`, in `train` and in
`test`; the number of rows must not change.

In [ ]:
# TODO: add the column weekly_footfall to train and to test, matched on
# store_name.
train = ...
test = ...

In [ ]:
assert len(train) == 1591 and len(test) == 409, "a join changed the rows"
assert train["weekly_footfall"].notna().all(), "a store has no footfall"
assert test["weekly_footfall"].notna().all(), "the test store has no footfall"

cv_mae["+ database"] = get_simple_baseline(train, drop_cols=DROP)
pd.Series(cv_mae, name="CV MAE").round(2)

Refit on the four sources and submit again. Both submissions stay on the
leaderboard, so the change in score is what the database added.

In [ ]:
cv, pred = get_simple_baseline(train, drop_cols=DROP, X_data_test=test)
submission = pd.DataFrame({"item_code": test.index, TARGET: pred})
assert submission[TARGET].notna().all()
submission.to_csv("submission.csv", index=False)
print(f"CV MAE: {cv:.2f}")

result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"],
                       agent_name="four-sources")
wait_for_score(result)